# GAWA: high-level overview
This notebook explains a character-level encoder/decoder that builds an **eword** (word vector) from a word. The encoder summarizes characters with position-aware weighting, then the decoder reconstructs the word from the eword. This is intended as a frontend to a Global Transformer.

## Quick notation
- Word length: $n$, character index: $i = 1..n$.
- Character embedding: $x_i$.
- Positional encoding: $p_i$.
- Fused representation: $c_i$.
- Position weights: $w_i$.
- Word vector: $\text{eword}$.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import json
from pathlib import Path

## 1) Character vocabulary
- Special tokens: $\langle\text{PAD}\rangle=0$, $\langle\text{BOS}\rangle=1$, $\langle\text{EOS}\rangle=2$, $\langle\text{UNK}\rangle=3$.
- Encoding: $\text{word} \to [id_1, id_2, \dots]$.
- Decoding stops at $\langle\text{EOS}\rangle$.


In [2]:
class CharVocab:
    def __init__(self):
        # Karakter dasar + spesial token
        chars = list("abcdefghijklmnopqrstuvwxyz")
        chars += list("abcdefghijklmnopqrstuvwxyz".upper())
        chars += list("0123456789")
        chars += list(".,!?'-_")
        
        self.PAD = 0
        self.BOS = 1  # begin of sequence
        self.EOS = 2  # end of sequence
        self.UNK = 3
        
        self.char2idx = {"<PAD>": 0, "<BOS>": 1, "<EOS>": 2, "<UNK>": 3}
        for i, c in enumerate(chars):
            self.char2idx[c] = i + 4
        
        self.idx2char = {v: k for k, v in self.char2idx.items()}
        self.vocab_size = len(self.char2idx)
    
    def encode(self, word: str) -> list[int]:
        return [self.char2idx.get(c, self.UNK) for c in word]
    
    def decode(self, indices: list[int]) -> str:
        result = []
        for idx in indices:
            if idx == self.EOS:
                break
            if idx not in (self.PAD, self.BOS, self.UNK):
                result.append(self.idx2char.get(idx, "?"))
        return "".join(result)

## 2) Gaussian positional encoding (fixed)
For position $i$ and dimension $j$:

$$
PE_{i,j} = \exp\left( - \frac{(i - \mu_j)^2}{2\sigma_j^2} \right),\quad
\mu_j = j,\quad \sigma_j = \sqrt{j}.
$$


In [3]:
class GaussianPositionalEncoding(nn.Module):
    def __init__(self, dim: int = 64):
        super().__init__()
        self.dim = dim
        # mu_j = j, sigma_j = sqrt(j) — fixed, tidak dilatih
        j = torch.arange(1, dim + 1, dtype=torch.float32)
        self.register_buffer("mu", j)
        self.register_buffer("sigma", torch.sqrt(j))
    
    def forward(self, positions: torch.Tensor) -> torch.Tensor:
        """
        positions: (batch, seq_len) — integer positions
        return:    (batch, seq_len, dim)
        """
        # (batch, seq_len, 1)
        pos = positions.unsqueeze(-1).float()
        # (1, 1, dim)
        mu = self.mu.view(1, 1, -1)
        sigma = self.sigma.view(1, 1, -1)
        
        # Gaussian: exp(-(i - mu_j)^2 / (2 * sigma_j^2))
        enc = torch.exp(-((pos - mu) ** 2) / (2 * sigma ** 2 + 1e-8))
        return enc  # (batch, seq_len, dim)

## 3) Gaussian prior for position weights
Parameters from the GAWA paper: $d=1.617$, $s_0=0.5$, $r=1.105$.

$$
\sigma_i = d - (d - s_0)\,e^{-ri},\quad
\text{importance}_i = \frac{1}{\sigma_i},\quad
w_i = \frac{\text{importance}_i}{\sum_k \text{importance}_k}.
$$


In [4]:
# ─────────────────────────────────────────
# 3. GAUSSIAN PRIOR UNTUK POSITIONAL WEIGHT
# ─────────────────────────────────────────

class GaussianPositionPrior(nn.Module):
    """
    Prior weight per posisi karakter dalam kata.
    d=1.617, s0=0.5, r=1.105 dari paper AYA.
    """
    def __init__(self, d=1.617, s0=0.5, r=1.105):
        super().__init__()
        self.d = d
        self.s0 = s0
        self.r = r
    
    def forward(self, n: int, device) -> torch.Tensor:
        """Return normalized prior weights shape (n,)"""
        i = torch.arange(1, n + 1, dtype=torch.float32, device=device)
        sigma = self.d - (self.d - self.s0) * torch.exp(-self.r * i)
        importance = 1.0 / sigma
        return importance / importance.sum()


## 4) GAWA Encoder: main equations
1. Character embedding: $x_i = \text{Emb}(\text{char}_i)$.
2. Positional encoding: $p_i = PE(i)$.
3. Fusion: $c_i = \text{MLP}([x_i; p_i])$.
4. Learnable adjustment:

$$
\Delta_i = \lambda\,\tanh(\text{MLP}([i, n]))
$$

5. Final weights:

$$
\tilde{w}_i = w_i (1 + \Delta_i),\quad
\alpha_i = \frac{\tilde{w}_i}{\sum_k \tilde{w}_k}
$$

6. Weighted pooling:

$$
\text{eword} = \sum_i \alpha_i c_i
$$

7. Output projection: $\text{eword} = W_{out}\,\text{eword}$.


## 5) GAWA Decoder (GRU + cross-attention)
At decoding step $t$:

$$
\text{input}_t = [\text{emb}(\text{char}_{t-1});\,\text{eword}]
$$

GRU produces hidden state $h_t$, then cross-attention to eword:

$$
\text{attn}_t = \text{Attn}(q=h_t,\,k=K(\text{eword}),\,v=V(\text{eword}))
$$

$$
\text{output}_t = h_t + \text{attn}_t,\quad
\text{logits}_t = W_{out}\,\text{output}_t
$$

Teacher forcing uses the ground-truth $\text{char}_{t-1}$ during training.


In [5]:
# ─────────────────────────────────────────
# 4. AYA ENCODER
# ─────────────────────────────────────────

class AYAEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        char_emb_dim: int = 64,
        pos_enc_dim: int = 64,
        hidden_dim: int = 256,
        output_dim: int = 768,
        lambda_adjust: float = 0.3,
    ):
        super().__init__()
        self.output_dim = output_dim
        self.lambda_adjust = lambda_adjust
        
        # Character embedding
        self.char_emb = nn.Embedding(vocab_size, char_emb_dim, padding_idx=0)
        
        # Gaussian positional encoding (fixed)
        self.pos_enc = GaussianPositionalEncoding(dim=pos_enc_dim)
        
        # Fusion MLP: char_emb + pos_enc → hidden
        fusion_in = char_emb_dim + pos_enc_dim
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fusion_in, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )
        
        # Gaussian prior
        self.prior = GaussianPositionPrior()
        
        # Learnable adjustment MLP: [position, word_len] → delta
        self.adjust_mlp = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )
        
        # Output projection → eword
        self.output_proj = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, char_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """
        char_ids: (batch, max_len) padded
        lengths:  (batch,) actual length per kata
        return:   (batch, output_dim) — eword
        """
        batch, max_len = char_ids.shape
        device = char_ids.device
        
        # 1. Character embedding
        x_emb = self.char_emb(char_ids)  # (batch, max_len, char_emb_dim)
        
        # 2. Gaussian positional encoding
        positions = torch.arange(1, max_len + 1, device=device).unsqueeze(0).expand(batch, -1)
        pos_enc = self.pos_enc(positions)  # (batch, max_len, pos_enc_dim)
        
        # 3. Fusion
        fused = torch.cat([x_emb, pos_enc], dim=-1)  # (batch, max_len, fusion_in)
        c = self.fusion_mlp(fused)  # (batch, max_len, hidden_dim)
        
        # 4. Dynamic positional weighting
        eword_list = []
        for b in range(batch):
            n = lengths[b].item()
            
            # Gaussian prior
            w_gauss = self.prior(n, device)  # (n,)
            
            # Learnable adjustment
            i_idx = torch.arange(1, n + 1, dtype=torch.float32, device=device)
            n_tensor = torch.full((n,), float(n), device=device)
            inp = torch.stack([i_idx, n_tensor], dim=-1)  # (n, 2)
            delta = self.lambda_adjust * torch.tanh(self.adjust_mlp(inp)).squeeze(-1)  # (n,)
            
            # Final weights
            w_final = w_gauss * (1 + delta)
            w_norm = w_final / (w_final.sum() + 1e-8)  # (n,)
            
            # Weighted pooling
            c_b = c[b, :n, :]  # (n, hidden_dim)
            eword = (w_norm.unsqueeze(-1) * c_b).sum(dim=0)  # (hidden_dim,)
            eword_list.append(eword)
        
        eword = torch.stack(eword_list, dim=0)  # (batch, hidden_dim)
        eword = self.output_proj(eword)  # (batch, output_dim)
        return eword


# ─────────────────────────────────────────
# 5. AYA DECODER
# ─────────────────────────────────────────

class AYADecoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        eword_dim: int = 768,
        char_emb_dim: int = 64,
        hidden_dim: int = 256,
        max_len: int = 32,
        num_layers: int = 2,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.hidden_dim = hidden_dim
        
        # Project eword → initial hidden state
        self.eword_proj = nn.Linear(eword_dim, hidden_dim)
        
        # Character embedding untuk decoder input
        self.char_emb = nn.Embedding(vocab_size, char_emb_dim, padding_idx=0)
        
        # GRU autoregressive decoder
        self.gru = nn.GRU(
            input_size=char_emb_dim + eword_dim,  # char + eword sebagai context
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        
        # Cross attention: query=hidden, key/value=eword
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            batch_first=True,
        )
        self.eword_key = nn.Linear(eword_dim, hidden_dim)
        self.eword_val = nn.Linear(eword_dim, hidden_dim)
        
        # Output projection → vocab logits
        self.output_proj = nn.Linear(hidden_dim, vocab_size)
    
    def forward(
        self,
        eword: torch.Tensor,
        target_ids: torch.Tensor = None,
        teacher_forcing: bool = True,
    ) -> torch.Tensor:
        """
        eword:      (batch, eword_dim)
        target_ids: (batch, max_len) — untuk teacher forcing saat training
        return:     (batch, max_len, vocab_size) — logits
        """
        batch = eword.shape[0]
        device = eword.device
        vocab = self.vocab_size
        
        # Initial hidden state dari eword
        h = self.eword_proj(eword).unsqueeze(0)  # (1, batch, hidden)
        h = h.repeat(self.gru.num_layers, 1, 1)  # (num_layers, batch, hidden)
        
        # eword sebagai context key/value untuk cross attention
        eword_k = self.eword_key(eword).unsqueeze(1)  # (batch, 1, hidden)
        eword_v = self.eword_val(eword).unsqueeze(1)  # (batch, 1, hidden)
        
        # Mulai dengan BOS token
        input_char = torch.full((batch,), 1, dtype=torch.long, device=device)  # BOS=1
        
        all_logits = []
        
        max_steps = target_ids.shape[1] if target_ids is not None else self.max_len
        
        for t in range(max_steps):
            # Char embedding
            char_emb = self.char_emb(input_char)  # (batch, char_emb_dim)
            
            # Concat dengan eword sebagai context langsung
            gru_in = torch.cat([char_emb, eword], dim=-1).unsqueeze(1)  # (batch, 1, ...)
            
            # GRU step
            out, h = self.gru(gru_in, h)  # out: (batch, 1, hidden)
            
            # Cross attention dengan eword
            attn_out, _ = self.cross_attn(out, eword_k, eword_v)  # (batch, 1, hidden)
            out = out + attn_out  # residual
            
            # Logits
            logits = self.output_proj(out.squeeze(1))  # (batch, vocab_size)
            all_logits.append(logits)
            
            # Next input
            if teacher_forcing and target_ids is not None:
                input_char = target_ids[:, t]
            else:
                input_char = logits.argmax(dim=-1)
        
        return torch.stack(all_logits, dim=1)  # (batch, max_steps, vocab_size)

## 6) Full model and dataset
- **GAWAModel** combines encoder and decoder.
- **WordDataset** builds targets: $[\langle\text{BOS}\rangle,\,\text{chars},\,\langle\text{EOS}\rangle]$.
- Inputs and targets are padded to $\text{max\_len}$.


In [6]:
# ─────────────────────────────────────────
# 6. AYA FULL MODEL
# ─────────────────────────────────────────

class AYAModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        char_emb_dim: int = 64,
        pos_enc_dim: int = 64,
        hidden_dim: int = 256,
        eword_dim: int = 768,
        max_word_len: int = 32,
    ):
        super().__init__()
        self.encoder = AYAEncoder(
            vocab_size=vocab_size,
            char_emb_dim=char_emb_dim,
            pos_enc_dim=pos_enc_dim,
            hidden_dim=hidden_dim,
            output_dim=eword_dim,
        )
        self.decoder = AYADecoder(
            vocab_size=vocab_size,
            eword_dim=eword_dim,
            char_emb_dim=char_emb_dim,
            hidden_dim=hidden_dim,
            max_len=max_word_len,
        )
    
    def forward(self, char_ids, lengths, target_ids=None, teacher_forcing=True):
        eword = self.encoder(char_ids, lengths)
        logits = self.decoder(eword, target_ids, teacher_forcing)
        return logits, eword
    
    def encode(self, char_ids, lengths):
        """Hanya encode → eword untuk dipakai di Global Transformer"""
        return self.encoder(char_ids, lengths)
    
    @torch.no_grad()
    def reconstruct(self, char_ids, lengths, vocab: CharVocab) -> list[str]:
        """Encode lalu decode balik → kata"""
        eword = self.encoder(char_ids, lengths)
        logits = self.decoder(eword, teacher_forcing=False)
        pred_ids = logits.argmax(dim=-1)  # (batch, max_len)
        results = []
        for ids in pred_ids:
            results.append(vocab.decode(ids.tolist()))
        return results


# ─────────────────────────────────────────
# 7. DATASET
# ─────────────────────────────────────────

class WordDataset(Dataset):
    def __init__(self, words: list[str], vocab: CharVocab, max_len: int = 32):
        self.vocab = vocab
        self.max_len = max_len
        # Filter kata yang terlalu panjang
        self.words = [w for w in words if 1 <= len(w) <= max_len - 2]
    
    def __len__(self):
        return len(self.words)
    
    def __getitem__(self, idx):
        word = self.words[idx]
        char_ids = self.vocab.encode(word)
        length = len(char_ids)
        
        # Target: BOS + chars + EOS
        target = [self.vocab.BOS] + char_ids + [self.vocab.EOS]
        
        # Pad
        pad_len = self.max_len - length
        char_ids_padded = char_ids + [self.vocab.PAD] * pad_len
        
        target_pad_len = (self.max_len + 1) - len(target)
        target_padded = target + [self.vocab.PAD] * target_pad_len
        
        return {
            "char_ids": torch.tensor(char_ids_padded, dtype=torch.long),
            "lengths": torch.tensor(length, dtype=torch.long),
            "target": torch.tensor(target_padded[:self.max_len], dtype=torch.long),
        }


## 7) Training objective
Character-level cross-entropy (ignoring PAD positions):

$$
\mathcal{L} = -\sum_t \log p(y_t = \text{target}_t)
$$

Accuracy is computed only on non-PAD characters. A cosine annealing scheduler is used.


## 8) Inference / sentence encoding
Each word is encoded into an eword vector. The result is:

$$
(\text{num\_words},\,\text{eword\_dim})
$$

This sequence is ready for a Global Transformer.


In [7]:

# ─────────────────────────────────────────
# 8. TRAINING LOOP
# ─────────────────────────────────────────

def train_aya(
    words: list[str],
    epochs: int = 20,
    batch_size: int = 256,
    lr: float = 1e-3,
    eword_dim: int = 768,
    save_path: str = "aya_checkpoint.pt",
    device: str = None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training di: {device}")
    
    # Vocab
    vocab = CharVocab()
    print(f"Vocab size: {vocab.vocab_size}")
    
    # Dataset
    dataset = WordDataset(words, vocab)
    print(f"Total kata: {len(dataset)}")
    
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(device == "cuda"),
    )
    
    # Model
    model = AYAModel(
        vocab_size=vocab.vocab_size,
        eword_dim=eword_dim,
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # Loss — ignore PAD
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.PAD)
    
    best_loss = float("inf")
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        correct = 0
        total_chars = 0
        
        for batch in loader:
            char_ids = batch["char_ids"].to(device)
            lengths = batch["lengths"].to(device)
            target = batch["target"].to(device)
            
            optimizer.zero_grad()
            
            # Forward — teacher forcing saat training
            logits, eword = model(
                char_ids, lengths,
                target_ids=target,
                teacher_forcing=True,
            )
            
            # Loss: logits (batch, max_len, vocab) vs target (batch, max_len)
            loss = criterion(
                logits.reshape(-1, vocab.vocab_size),
                target.reshape(-1),
            )
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            
            # Accuracy per karakter
            mask = target != vocab.PAD
            pred = logits.argmax(dim=-1)
            correct += (pred[mask] == target[mask]).sum().item()
            total_chars += mask.sum().item()
        
        scheduler.step()
        
        avg_loss = total_loss / len(loader)
        acc = correct / total_chars * 100
        
        print(f"Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | Char Acc: {acc:.2f}%")
        
        # Simpan checkpoint terbaik
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "vocab": vocab.char2idx,
                "config": {
                    "vocab_size": vocab.vocab_size,
                    "eword_dim": eword_dim,
                },
            }, save_path)
        
        # Sample rekonstruksi tiap 5 epoch
        if epoch % 5 == 0:
            model.eval()
            sample_words = words[:5]
            sample_dataset = WordDataset(sample_words, vocab)
            sample_batch = {
                "char_ids": torch.stack([sample_dataset[i]["char_ids"] for i in range(len(sample_dataset))]).to(device),
                "lengths": torch.stack([sample_dataset[i]["lengths"] for i in range(len(sample_dataset))]).to(device),
            }
            reconstructed = model.reconstruct(
                sample_batch["char_ids"],
                sample_batch["lengths"],
                vocab,
            )
            print("\n  Sample rekonstruksi:")
            for orig, rec in zip(sample_words, reconstructed):
                status = "✓" if orig.lower() == rec.lower() else "✗"
                print(f"    {status} '{orig}' → '{rec}'")
            print()
    
    print(f"\nTraining selesai! Best loss: {best_loss:.4f}")
    print(f"Checkpoint disimpan di: {save_path}")
    return model, vocab


# ─────────────────────────────────────────
# 9. INFERENCE / ENCODE KALIMAT
# ─────────────────────────────────────────

def encode_sentence(
    sentence: str,
    model: AYAModel,
    vocab: CharVocab,
    device: str = "cpu",
    max_word_len: int = 32,
) -> torch.Tensor:
    """
    Encode kalimat → sequence of eword vectors
    Ini yang masuk ke Global Transformer nantinya
    
    return: (num_words, eword_dim)
    """
    words = sentence.lower().split()
    model.eval()
    
    ewords = []
    with torch.no_grad():
        for word in words:
            char_ids = vocab.encode(word)[:max_word_len - 2]
            length = len(char_ids)
            pad_len = max_word_len - length
            char_ids_padded = char_ids + [vocab.PAD] * pad_len
            
            char_tensor = torch.tensor([char_ids_padded], dtype=torch.long, device=device)
            len_tensor = torch.tensor([length], dtype=torch.long, device=device)
            
            eword = model.encode(char_tensor, len_tensor)  # (1, eword_dim)
            ewords.append(eword.squeeze(0))
    
    return torch.stack(ewords, dim=0)  # (num_words, eword_dim)


# ─────────────────────────────────────────
# 10. MAIN — CONTOH PENGGUNAAN
# ─────────────────────────────────────────

if __name__ == "__main__":
    
    # Contoh kata — ganti dengan dataset kamu yang lebih besar
    # Idealnya 500rb - 1jt kata unik dari kamus / Wikipedia
    sample_words = [
        "saya", "kamu", "dia", "kita", "mereka",
        "makan", "memakan", "makanan", "dimakan",
        "minum", "minuman", "meminum",
        "jalan", "berjalan", "perjalanan",
        "buku", "membaca", "bacaan",
        "tulis", "menulis", "tulisan", "penulis",
        "rumah", "perumahan",
        "sekolah", "bersekolah",
        "kerja", "bekerja", "pekerjaan", "pekerja",
        "main", "bermain", "permainan",
        "bicara", "berbicara", "pembicaraan",
        "lari", "berlari", "pelari",
        "tidur", "tertidur",
        "bangun", "membangun", "bangunan",
        "beli", "membeli", "pembelian",
        "jual", "menjual", "penjual", "penjualan",
        "indonesia", "bahasa", "kata", "kalimat",
        "komputer", "program", "data", "model",
        "latih", "melatih", "pelatihan", "terlatih",
        "belajar", "pembelajaran", "pelajar",
        "menggunakan", "digunakan", "penggunaan",
        "membuat", "pembuatan", "dibuat",
        "pengembangan", "mengembangkan",
        "pembaharuan", "memperbarui",
    ]
    
    print("=" * 50)
    print("AYA: Automatic Yield Abstraction")
    print("Morphological Character-Level Encoder/Decoder")
    print("=" * 50)
    
    # Training
    model, vocab = train_aya(
        words=sample_words,
        epochs=100,
        batch_size=16,       # kecil karena sample kecil
        lr=1e-3,
        eword_dim=768,       # kompatibel dengan transformer standar
        save_path="aya_checkpoint.pt",
    )
    
    # Test encode kalimat → siap masuk Global Transformer
    print("\n" + "=" * 50)
    print("Test: Encode kalimat → eword vectors")
    print("=" * 50)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sentence = "saya sedang belajar membuat model bahasa"
    ewords = encode_sentence(sentence, model, vocab, device)
    
    print(f"\nKalimat: '{sentence}'")
    print(f"Jumlah kata: {len(sentence.split())}")
    print(f"Shape eword sequence: {ewords.shape}")
    print(f"  → {ewords.shape[0]} kata × {ewords.shape[1]} dimensi")
    print(f"\nIni yang masuk ke Global Transformer!")
    print(f"Sequence length jauh lebih pendek dari BPE tokenizer.")

AYA: Automatic Yield Abstraction
Morphological Character-Level Encoder/Decoder
Training di: cuda
Vocab size: 73
Total kata: 76
Total parameters: 2,410,058
Epoch   1/100 | Loss: 3.7703 | Char Acc: 19.19%
Epoch   2/100 | Loss: 2.6825 | Char Acc: 27.56%
Epoch   3/100 | Loss: 2.3682 | Char Acc: 30.88%
Epoch   4/100 | Loss: 2.1857 | Char Acc: 34.20%
Epoch   5/100 | Loss: 2.0250 | Char Acc: 37.37%

  Sample rekonstruksi:
    ✗ 'saya' → 'eeaa'
    ✗ 'kamu' → 'eeaa'
    ✗ 'dia' → 'eia'
    ✗ 'kita' → 'eela'
    ✗ 'mereka' → 'peeraa'

Epoch   6/100 | Loss: 1.8647 | Char Acc: 39.39%
Epoch   7/100 | Loss: 1.6756 | Char Acc: 50.65%
Epoch   8/100 | Loss: 1.4914 | Char Acc: 53.82%
Epoch   9/100 | Loss: 1.3422 | Char Acc: 57.14%
Epoch  10/100 | Loss: 1.2200 | Char Acc: 62.63%

  Sample rekonstruksi:
    ✗ 'saya' → 'aaa'
    ✗ 'kamu' → 'man'
    ✗ 'dia' → 'dat'
    ✗ 'kita' → 'dat'
    ✗ 'mereka' → 'membemam'

Epoch  11/100 | Loss: 1.0964 | Char Acc: 68.98%
Epoch  12/100 | Loss: 1.0571 | Char Acc: 65.